In [ ]:
import dolfinx
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver, self_supervised_train

from dolfinx.io import XDMFFile
from mpi4py import MPI
from SPDE_problems import int_to_prblm


tset = graph_dataset(f"data/training_set_globalizer_wedge/input_values")

class batched_loss_fn():
    def __init__(self, set):
        self.fsl = {}
        for G in set:
            num = G.mesh_id[0]
            with XDMFFile(MPI.COMM_WORLD, f"data/training_set/mesh_files/mesh_{G.mesh_id[0]}.xdmf", "r") as xdmf:
                mesh = xdmf.read_mesh(name="mesh")


            fs = int_to_prblm(idx=G.prblm_id, mesh=mesh)
            self.fsl[int(G.mesh_id)] = fem_solver(fs)

    def __call__(self, ptr, idx, y):
        loss_vals = [self.fsl[int(idx[i])](y[ptr[i]:ptr[i+1]] ) for i in range(len(ptr)-1)]
        return torch.stack(loss_vals).sum()
    
loss_fn = batched_loss_fn(tset)


class gat(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = tg.nn.models.GAT(
            in_channels=12, 
            hidden_channels=5, 
            num_layers=10, 
            out_channels=1, 
            v2=True, 
            #dropout=0., 
            act=torch.relu, 
            #norm=torch_geometric.nn.norm.LayerNorm(1),
            add_self_loops=False,
            residual=False
        )
        self.a = torch.tensor([0.1], dtype=torch.float32)
        self.b = torch.tensor([0.1], dtype=torch.float32)

    def forward(self, data) -> torch.Tensor:
        x, edge_index, delta_uh, D_uh = data.x, data.edge_index, data.delta_uh, data.D_uh
        h = self.model(
            x=x,
            edge_index=edge_index
        )
        upper = torch.sigmoid(self.a)*delta_uh + torch.sigmoid(self.b)*D_uh
        return upper*torch.sigmoid(h)
    


Processing...
Done!


In [4]:
batch_size = 1
loader = train_loader(batch_size=batch_size, set=tset)


In [2]:
model=gat()


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.8, patience=50)
curr_loss = 10



In [5]:

for i in range(1000):
    loss = self_supervised_train(model=model, loader=loader,loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if curr_loss > loss:    
        print(f"iteration {i}: new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/GATv2_self_supervised_globalizer_wedge.pth")
        #scheduler.step(loss)
    else:
        print(f"iteration {i}: {loss}")
        scheduler.step(loss)

        


iteration 0: new loss: 0.8455946346124014
iteration 1: new loss: 0.8455938994884491
iteration 2: new loss: 0.8455931577417586
iteration 3: new loss: 0.8455924325519137
iteration 4: new loss: 0.8455917172961764
iteration 5: new loss: 0.8455910020404391
iteration 6: new loss: 0.8455903000301785
iteration 7: new loss: 0.8455895980199178
iteration 8: new loss: 0.8455889225006104
iteration 9: new loss: 0.8455882370471954
iteration 10: new loss: 0.8455875681506263
iteration 11: new loss: 0.8455869091881646
iteration 12: new loss: 0.8455862469143338
iteration 13: new loss: 0.8455855912632413
iteration 14: new loss: 0.8455849488576254
iteration 15: new loss: 0.8455843064520094
iteration 16: new loss: 0.8455836872259775
iteration 17: new loss: 0.8455830646885766
iteration 18: new loss: 0.8455824487739139
iteration 19: new loss: 0.845581849416097
iteration 20: new loss: 0.8455812467469109
iteration 21: new loss: 0.8455806606345706
iteration 22: new loss: 0.8455800645881228
iteration 23: new loss